# Training snooper

Snapshot viewer for active or completed KRG-alsDenoise training logs. Each refresh
uses the same `plot_training_history()` plots as the HPC notebook: L1, weighted
LCL, optional MSE/SSIM, and total loss with the logged early-stopping curve.

The notebook reads log data only. It does not load checkpoints, use a GPU,
initialize reconstruction state, or control the training process. Exports go
to a separate directory for each snapshot.

## Setup and concurrent use

Open this notebook in a **separate kernel** from the training notebook. On NERSC,
select **ALS CT + N2I notebook** and start a new kernel; attaching to the busy
training kernel would queue the viewer cells behind training.

The viewer requires Python 3.8 or newer, the `krg-ct-workflow` package, NumPy,
Matplotlib and IPython. The documented ALS environment already supplies these
dependencies. From the repository checkout, installation in that environment is:

```bash
conda activate "$HOME/microct-n2i-envs/als832"
python -m pip install --no-deps -e "$HOME/microct-n2i-source/KRG-alsDenoise/workflow"
```

See [NERSC setup](../docs/SETUP_NERSC.md) for a new installation. The viewer needs
filesystem access to the log and a writable snapshot destination; no GPU
allocation is required for plotting. The same compute Jupyter session can host
both notebook kernels.

Run cells 1–3 once. Rerun cell 3 for a fresh snapshot; cell 4 shows the captured
log tail. No background polling runs between refreshes.

In [ ]:
# 1. Import the log viewer
from pathlib import Path
import json
import krg_workflow
from krg_workflow.history import snapshot_training_history
from IPython.display import Image, display

print('Workflow:', krg_workflow.__version__, '| source:', krg_workflow.__file__)


## Log selection

Set `TRAIN_LOG` to the complete `train.log` path printed by the training launcher.
The usual NERSC layout is
`$PSCRATCH/noise2inverse360/<scan>/<run>/logs/train.log`. In Python, use an
absolute path or `Path.home()`; literal `$PSCRATCH` text is not expanded by `Path`.

`SNAPSHOT_ROOT=None` creates `training_snooper/` beside the log. Set an absolute
`Path` to store snapshots elsewhere. Each refresh creates a unique subdirectory
containing PNG, SVG, CSV and `snapshot.json`; training's `training_history/`
exports are not replaced.

`MOVING_AVERAGE_EPOCHS` controls display smoothing. The stopping curve comes
directly from the log and retains the training run's actual policy.

Plots use a linear epoch axis and logarithmic loss axes with major and minor gridlines.
Dots mark successive raw loss minima separately for each curve after warmup;
the epoch containing the last warmup updates is also excluded. Nonpositive
values remain in CSV exports but are omitted from logarithmic plots. Logs
without warmup status are plotted without record-low markers.

In [ ]:
# 2. Select the training log and display settings
TRAIN_LOG = Path('/absolute/path/to/run/logs/train.log')
SNAPSHOT_ROOT = None
MOVING_AVERAGE_EPOCHS = 10

print('Training log:', TRAIN_LOG)
print('Snapshot root:', SNAPSHOT_ROOT or TRAIN_LOG.parent/'training_snooper')


## Snapshot

Each refresh reads up to the file size observed at open and drops an unfinished
final line. Plots include records with complete L1/LCL summaries; optional
metrics appear when available. Resumed epochs use the latest complete record.
An empty or missing log produces a status message until records are available.

The capture time and latest parsed epoch describe the log snapshot. They do not
confirm that the training process is still running. Last modification time and
the optional log tail help distinguish recent output from an unchanged log.

In [ ]:
# 3. Refresh the training snapshot
snapshot = snapshot_training_history(TRAIN_LOG, output_root=SNAPSHOT_ROOT,
                                     moving_window=MOVING_AVERAGE_EPOCHS, show=False)
summary = {key: snapshot[key] for key in ('status', 'captured_utc', 'log_modified_utc',
           'epochs', 'last_epoch', 'best_l1', 'best_lcl', 'omitted_tail_bytes') if key in snapshot}
print(json.dumps(summary, indent=2))
if snapshot['status'] == 'saved':
    display(Image(filename=snapshot['paths']['png']))
    print('Exports:', json.dumps(snapshot['paths'], indent=2))


In [ ]:
# 4. Show the captured log tail (optional)
print('\n'.join(snapshot['log_tail']) or 'No complete log lines in this snapshot.')
